# Linear Regression Regularized Modelado

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.metrics import median_absolute_error, mean_absolute_percentage_error
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LassoCV, RidgeCV, ElasticNet
import pickle

## Obtención de datos

In [2]:
X_test = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/linear-regression-reg-X-test.csv', 
                     index_col='COUNTY_NAME')
X_train = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/linear-regression-reg-X-train.csv',
                      index_col='COUNTY_NAME')
y_test = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/linear-regression-reg-y-test.csv',
                     index_col='COUNTY_NAME')
y_train = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/linear-regression-reg-y-train.csv',
                      index_col='COUNTY_NAME')

>Recogemos los datos generados en el EDA anteriormente creado

## Definimos get_metrics

In [3]:
def get_metrics(y_predict_test, y_test, y_predict_train, y_train):
    metrics_train = (r2_score(y_train, y_predict_train),
                     median_absolute_error(y_train, y_predict_train),
                     mean_absolute_percentage_error(y_train, y_predict_train) * 100,
                     mean_squared_error(y_train, y_predict_train),
                     root_mean_squared_error(y_train, y_predict_train))
    metrics_test = (r2_score(y_test, y_predict_test),
                    median_absolute_error(y_test, y_predict_test),
                    mean_absolute_percentage_error(y_test, y_predict_test) * 100,
                    mean_squared_error(y_test, y_predict_test),
                    root_mean_squared_error(y_test, y_predict_test))
    metrics_diff = list(map(lambda x: x[1] - x[0], zip(metrics_train, metrics_test)))
    return pd.DataFrame(data=[metrics_train, metrics_test, metrics_diff],
                        columns=['R2', 'MAE', 'MAPE', 'MSE', 'RMSE'],
                        index=['Train set', 'Test set', 'Difference'])

## Regresión lineal básica

In [4]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_y_pred_train = lr_model.predict(X_train)
lr_y_pred_test = lr_model.predict(X_test)

get_metrics(lr_y_pred_test, y_test, lr_y_pred_train, y_train)

,R2,MAE,MAPE,MSE,RMSE
Train set,0.999673,0.005463,3.408581e+12,0.002343,0.048403
Test set,0.999671,0.005751,7.178704e+13,0.002447,0.049463
Difference,-0.000002,0.000288,6.837846e+13,0.000104,0.001060


## Lasso

In [5]:
lasso_model = Lasso(alpha=0.5, max_iter=400, random_state=25)  # modelo
lasso_model.fit(X_train, y_train)  # entrenamiento

lasso_y_pred_test = lasso_model.predict(X_test)  # Predicción en test
lasso_y_pred_train = lasso_model.predict(X_train)  # Predicción en train
get_metrics(lasso_y_pred_test, y_test, lasso_y_pred_train, y_train)  # Métricas

,R2,MAE,MAPE,MSE,RMSE
Train set,0.661981,0.902784,4.191921e+15,2.421601,1.556149
Test set,0.637452,0.955854,8.017900e+15,2.698975,1.642856
Difference,-0.024530,0.053070,3.825979e+15,0.277374,0.086706


### LassoCV

In [6]:
lasso_cv_model = (LassoCV(alphas=np.logspace(-6, 6, 10),
                         cv=5,
                         random_state=25,
                         n_jobs=-1).fit(X_train, y_train))

lasso_cv_y_pred_test = lasso_cv_model.predict(X_test)  # Predicción en test
lasso_cv_y_pred_train = lasso_cv_model.predict(X_train)  # Predicción en train
get_metrics(lasso_cv_y_pred_test, y_test, lasso_cv_y_pred_train, y_train)  # Métricas

/home/vscode/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:1705: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/vscode/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.995e+00, tolerance: 1.449e+00
  model = cd_fast.enet_coordinate_descent_gram(
/home/vscode/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.798e+00, tolerance: 1.473e+00
  model = cd_fast.enet_coordinate_

,R2,MAE,MAPE,MSE,RMSE
Train set,0.999670,0.003752,5.844133e+12,0.002366,0.048642
Test set,0.999679,0.003884,2.861227e+13,0.002388,0.048862
Difference,0.000010,0.000132,2.276813e+13,0.000022,0.000221


## Ridge

In [7]:
ridge_model = Ridge(alpha=0.0001,
                    max_iter=5000,
                    random_state=25).fit(X_train, y_train)
ridge_y_pred_test = ridge_model.predict(X_test)  # Predicción en test
ridge_y_pred_train = ridge_model.predict(X_train)  # Predicción en test

get_metrics(ridge_y_pred_test, y_test, ridge_y_pred_train, y_train)  # Métricas

,R2,MAE,MAPE,MSE,RMSE
Train set,9.996725e-01,0.005048,3.132677e+12,0.002346,0.048435
Test set,9.996733e-01,0.005098,7.155107e+13,0.002432,0.049318
Difference,7.331792e-07,0.000050,6.841840e+13,0.000086,0.000883


### RidgeCV

In [8]:
ridge_cv_model = RidgeCV(alphas=np.logspace(-6, 6, 10),
                         cv=5).fit(X_train, y_train)

ridge_cv_y_pred_test = ridge_cv_model.predict(X_test)  # Predicción en test
ridge_cv_y_pred_train = ridge_cv_model.predict(X_train)  # Predicción en train
get_metrics(ridge_cv_y_pred_test, y_test, ridge_cv_y_pred_train, y_train)

,R2,MAE,MAPE,MSE,RMSE
Train set,9.996725e-01,0.005018,3.091597e+12,0.002346,0.048436
Test set,9.996734e-01,0.004966,7.131098e+13,0.002431,0.049310
Difference,8.518100e-07,-0.000052,6.821939e+13,0.000085,0.000874


## ElasticNet

In [9]:
elasticnet_model = ElasticNet(alpha=0.0001, max_iter=400, random_state=25).fit(X_train, y_train)

elasticnet_y_pred_test = elasticnet_model.predict(X_test)
elasticnet_y_pred_train = elasticnet_model.predict(X_train)
get_metrics(elasticnet_y_pred_test, y_test, elasticnet_y_pred_train, y_train)

/home/vscode/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.800e+00, tolerance: 1.800e+00
  model = cd_fast.enet_coordinate_descent(


,R2,MAE,MAPE,MSE,RMSE
Train set,0.999656,0.006156,4.504349e+12,0.002464,0.049639
Test set,0.999623,0.006466,1.401249e+13,0.002807,0.052982
Difference,-0.000033,0.000310,9.508137e+12,0.000343,0.003343


## Comparación y optimización

>Voy a comparar los R2 de todos los modelos tanto de train set como de test set para ver cual de los 6 modelos es el que mejor funciona

### R2
|Set  |Regresión Lineal|Lasso   |LassoCV |Ridge    |RidgeCV  |ElasticNet|
|-----|----------------|--------|--------|---------|---------|----------|
|Train|0.999673        |0.661981|0.999670|0.9996725|0.9996725|0.999656  |
|Test |0.999671        |0.637452|0.999679|0.9996733|0.9996734|0.999623  |


>Vemos que el mejor resultado con el train de R2 lo da *Regresión Lineal* y el mejor resultado con Test lo da *LassoCV*, para ver con cual de nos dos nos quedamos vamos a comparar las diferencias que tienen entre ambos y nos quedaremos el que tenga menor diferencia

#### Diferencias
>La diferencia en regresión lineal es de -0.000002 mientras que la de LassoCV es de 0.000010 por lo que la diferencia de la regresión lineal es más cercana a 0 y nos quedaremos con eso y por tanto como modelo que guardaremos es el de la regresión lineal básica

In [10]:
with open('/workspaces/adamcn10-intro-ml/models/linear-regression-reg-model.pkl', 'wb') as file:
    pickle.dump(lr_model, file)